# Evolução dos votos por candidato

Este notebook reconstrói a evolução da apuração do 2º turno de 2022 usando
`DT_BU_RECEBIDO`, que representa o recebimento do boletim de urna. Cada candidato
aparece em uma linha própria.

**Critérios:** somente votos nominais (`CD_TIPO_VOTAVEL = 1`); cada horário distinto
de `DT_BU_RECEBIDO` forma uma atualização; eventos recebidos no mesmo segundo
permanecem no mesmo lote. Os votos são acumulados em ordem crescente de recebimento,
sem inventar uma ordem entre boletins que possuem exatamente o mesmo horário.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

pio.renderers.default = "plotly_mimetype"

def localizar_raiz(inicio: Path) -> Path:
    inicio = inicio.resolve()
    for pasta in (inicio, *inicio.parents):
        if (pasta / "04_bi" / "dados" / "fato_votos").exists():
            return pasta
    raise FileNotFoundError("Não foi possível localizar 04_bi/dados/fato_votos")

RAIZ = localizar_raiz(Path.cwd())
FATOS = RAIZ / "04_bi" / "dados" / "fato_votos" / "**" / "*.parquet"
DIM_VOTAVEL = RAIZ / "04_bi" / "dados" / "dimensoes" / "dim_votavel.parquet"

pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")

CONFIG_GRAFICO = {
    "scrollZoom": True,
    "displayModeBar": True,
    "displaylogo": False,
    "responsive": True,
    "doubleClick": "reset+autosize",
    "toImageButtonOptions": {"format": "png", "scale": 2},
}

def configurar_replay(figura, titulo, eixo_y):
    figura.update_layout(
        title={"text": titulo, "x": 0.02, "xanchor": "left"},
        height=650,
        template="plotly_white",
        hovermode="closest",
        hoverdistance=30,
        spikedistance=-1,
        dragmode="zoom",
        legend={"title": {"text": "Candidato"}, "orientation": "h", "y": 1.08},
        margin={"l": 75, "r": 35, "t": 105, "b": 85},
        xaxis={
            "title": "Ordem original das atualizações",
            "type": "linear",
            "tickformat": ",d",
            "rangeslider": {"visible": True, "thickness": 0.10},
            "showspikes": True,
            "spikemode": "across",
            "spikesnap": "data",
        },
        yaxis={
            "title": eixo_y,
            "rangemode": "tozero",
            "separatethousands": True,
            "showspikes": True,
            "spikemode": "across",
            "spikesnap": "data",
        },
    )
    return figura

print(f"Raiz localizada: {RAIZ}")

Raiz localizada: /home/administrador/Downloads/Apuracoes/2t2022


## Parâmetros

O padrão mostra Presidente no Brasil inteiro. Para analisar uma UF, altere `UF`
para uma sigla como `"SP"`. Para Governador, use `CARGO = 3` e selecione uma UF.

In [2]:
CARGO = 1  # 1 = Presidente; 3 = Governador
UF = None  # None = Brasil inteiro; exemplo: "SP"

if CARGO == 3 and UF is None:
    print("Aviso: para Governador, selecione uma UF para evitar misturar disputas estaduais.")

## Extração e preparação da série temporal

In [3]:
con = duckdb.connect()
caminho_fatos = str(FATOS).replace("'", "''")
caminho_dim = str(DIM_VOTAVEL).replace("'", "''")

filtro_uf = "AND v.sg_uf = ?" if UF else ""
parametros = [CARGO] + ([UF.upper()] if UF else [])

consulta_instantes = f"""
    SELECT DISTINCT v.dt_bu_recebido AS instante
    FROM read_parquet('{caminho_fatos}', hive_partitioning=true) v
    WHERE v.cd_cargo = ?
      {filtro_uf}
    ORDER BY instante
"""
instantes_originais = con.execute(consulta_instantes, parametros).df()["instante"]

consulta = f"""
    SELECT
        v.dt_bu_recebido AS instante,
        concat(d.nm_votavel, ' (', d.nr_votavel, ')') AS candidato,
        sum(v.qt_votos)::BIGINT AS votos_no_lote
    FROM read_parquet('{caminho_fatos}', hive_partitioning=true) v
    JOIN read_parquet('{caminho_dim}') d USING (id_votavel)
    WHERE v.cd_cargo = ?
      AND d.cd_tipo_votavel = 1
      {filtro_uf}
    GROUP BY instante, candidato
    ORDER BY instante, candidato
"""

lotes = con.execute(consulta, parametros).df()
if lotes.empty:
    raise ValueError("A seleção não retornou votos. Revise CARGO e UF.")

incrementos = (
    lotes.pivot_table(index="instante", columns="candidato", values="votos_no_lote", aggfunc="sum")
    .fillna(0)
    .sort_index()
)
incrementos = incrementos.reindex(pd.DatetimeIndex(instantes_originais), fill_value=0)
incrementos.index.name = "instante"
acumulado = incrementos.cumsum()
participacao = acumulado.div(acumulado.sum(axis=1), axis=0).mul(100).fillna(0)
ordem_atualizacao = pd.RangeIndex(1, len(incrementos) + 1, name="atualizacao")
horarios_texto = incrementos.index.strftime("%d/%m/%Y %H:%M:%S")

print(f"Período: {acumulado.index.min()} a {acumulado.index.max()}")
print(f"Atualizações reproduzidas em ordem: {len(acumulado):,}")
print(f"Candidatos: {', '.join(acumulado.columns)}")

Período: 2022-10-30 01:20:48 a 2022-10-31 00:17:31
Atualizações reproduzidas em ordem: 11,388
Candidatos: JAIR BOLSONARO (22), LULA (13)


## Validação da série

A célula abaixo confere novamente o total diretamente no fato e garante que nenhuma
linha acumulada diminua. Se houver divergência, a execução para com erro.

In [4]:
consulta_total = f"""
    SELECT sum(v.qt_votos)::BIGINT
    FROM read_parquet('{caminho_fatos}', hive_partitioning=true) v
    JOIN read_parquet('{caminho_dim}') d USING (id_votavel)
    WHERE v.cd_cargo = ?
      AND d.cd_tipo_votavel = 1
      {filtro_uf}
"""
total_fato = con.execute(consulta_total, parametros).fetchone()[0]
total_serie = int(incrementos.to_numpy().sum())
monotono = bool(acumulado.diff().fillna(acumulado).ge(0).all().all())
percentuais_fecham = bool(participacao.sum(axis=1).sub(100).abs().lt(1e-9).all())
ordem_cronologica = bool(incrementos.index.is_monotonic_increasing)
instantes_unicos = not bool(incrementos.index.has_duplicates)

assert total_serie == total_fato, f"Divergência: série={total_serie}, fato={total_fato}"
assert monotono, "Ao menos uma série acumulada diminuiu."
assert percentuais_fecham, "Os percentuais por atualização não fecham em 100%."
assert ordem_cronologica, "As atualizações não estão em ordem cronológica."
assert instantes_unicos, "Existem atualizações duplicadas no eixo sequencial."

validacao = pd.DataFrame({
    "controle": [
        "Total da série = total do fato",
        "Séries acumuladas não diminuem",
        "Percentuais fecham em 100% a cada atualização",
        "Atualizações seguem DT_BU_RECEBIDO",
        "Um ponto por segundo/lote nacional",
    ],
    "resultado": ["OK", "OK", "OK", "OK", "OK"],
    "valor": [
        f"{total_serie:,}",
        monotono,
        percentuais_fecham,
        ordem_cronologica,
        instantes_unicos,
    ],
})
display(validacao)

,controle,resultado,valor
0,Total da série = total do fato,OK,"118,552,353"
1,Séries acumuladas não diminuem,OK,True
2,Percentuais fecham em 100% a cada atualização,OK,True
3,Atualizações seguem DT_BU_RECEBIDO,OK,True
4,Um ponto por segundo/lote nacional,OK,True


## Resultado final da seleção

In [5]:
resultado = acumulado.iloc[-1].sort_values(ascending=False).rename("votos").to_frame()
resultado["percentual_validos"] = resultado["votos"] / resultado["votos"].sum() * 100
resultado["votos_formatados"] = resultado["votos"].map(lambda x: f"{int(x):,}".replace(",", "."))
display(resultado[["votos_formatados", "percentual_validos"]])

,votos_formatados,percentual_validos
candidato,,
LULA (13),60.345.999,50.90
JAIR BOLSONARO (22),58.206.354,49.10


## Replay da apuração — porcentagem por atualização

Este é o gráfico principal. O eixo horizontal não representa minutos nem intervalos
artificiais: ele numera os lotes nacionais na ordem crescente de `DT_BU_RECEBIDO`.
O ponto 1 é a primeira atualização disponível, o ponto 2 é a seguinte, e assim por
diante. Cada linha mostra a porcentagem acumulada de votos válidos do candidato após
aquela atualização.

**Zoom retangular:** a lupa já fica selecionada. Clique em um ponto vazio, mantenha o
botão do mouse pressionado e desenhe um retângulo sobre qualquer região. A faixa
inferior permite escolher livremente qualquer sequência de atualizações. Um duplo
clique restaura a visão completa.

Ao passar o mouse sobre a linha, o gráfico mostra o número da atualização, horário
original, porcentagem, votos acumulados e votos que entraram naquele lote.

In [6]:
cores = {"LULA (13)": "#D62728", "JAIR BOLSONARO (22)": "#1F77B4"}
local = "Brasil" if UF is None else UF.upper()
fig = go.Figure()
for candidato in participacao.columns:
    detalhes = pd.concat(
        [acumulado[candidato], incrementos[candidato]],
        axis=1,
    ).to_numpy()
    fig.add_trace(go.Scatter(
        x=ordem_atualizacao,
        y=participacao[candidato],
        text=horarios_texto,
        customdata=detalhes,
        name=candidato,
        mode="lines+markers",
        line={"width": 2.2, "color": cores.get(candidato), "shape": "hv"},
        marker={"size": 3, "color": cores.get(candidato)},
        hovertemplate=(
            "<b>%{fullData.name}</b><br>"
            "Atualização: %{x:,.0f}<br>"
            "Horário: %{text}<br>"
            "Percentual dos válidos: %{y:.4f}%<br>"
            "Quantidade acumulada: %{customdata[0]:,.0f} votos<br>"
            "Votos neste lote: +%{customdata[1]:,.0f}"
            "<extra></extra>"
        ),
    ))

configurar_replay(
    fig,
    f"Replay da apuração: percentual após cada atualização — {local}",
    "Participação nos votos válidos (%)",
)
fig.update_yaxes(ticksuffix="%", rangemode="normal")
fig.show(config=CONFIG_GRAFICO)

## Quantidade acumulada por atualização

A mesma sequência original, agora mostrando a quantidade acumulada de votos nominais.
O horário e o percentual continuam disponíveis ao passar o mouse sobre cada linha.

In [7]:
fig = go.Figure()
for candidato in acumulado.columns:
    detalhes = pd.concat(
        [participacao[candidato], incrementos[candidato]],
        axis=1,
    ).to_numpy()
    fig.add_trace(go.Scatter(
        x=ordem_atualizacao,
        y=acumulado[candidato],
        text=horarios_texto,
        customdata=detalhes,
        name=candidato,
        mode="lines+markers",
        line={"width": 2.2, "color": cores.get(candidato), "shape": "hv"},
        marker={"size": 3, "color": cores.get(candidato)},
        hovertemplate=(
            "<b>%{fullData.name}</b><br>"
            "Atualização: %{x:,.0f}<br>"
            "Horário: %{text}<br>"
            "Quantidade acumulada: %{y:,.0f} votos<br>"
            "Percentual dos válidos: %{customdata[0]:.4f}%<br>"
            "Votos neste lote: +%{customdata[1]:,.0f}"
            "<extra></extra>"
        ),
    ))

configurar_replay(
    fig,
    f"Votos acumulados após cada atualização — {local}",
    "Votos nominais acumulados",
)
fig.update_yaxes(tickformat=",.0f")
fig.show(config=CONFIG_GRAFICO)

## Votos adicionados em cada atualização

Este gráfico mostra o tamanho de cada lote na mesma ordem do replay. Ele ajuda a
investigar se mudanças de porcentagem coincidem com lotes maiores ou menores.

In [8]:
fig = go.Figure()
for candidato in incrementos.columns:
    detalhes = pd.concat(
        [acumulado[candidato], participacao[candidato]],
        axis=1,
    ).to_numpy()
    fig.add_trace(go.Scatter(
        x=ordem_atualizacao,
        y=incrementos[candidato],
        text=horarios_texto,
        customdata=detalhes,
        name=candidato,
        mode="lines+markers",
        line={"width": 1.2, "color": cores.get(candidato)},
        marker={"size": 3, "color": cores.get(candidato)},
        hovertemplate=(
            "<b>%{fullData.name}</b><br>"
            "Atualização: %{x:,.0f}<br>"
            "Horário: %{text}<br>"
            "Votos neste lote: +%{y:,.0f}<br>"
            "Quantidade acumulada: %{customdata[0]:,.0f} votos<br>"
            "Percentual dos válidos: %{customdata[1]:.4f}%"
            "<extra></extra>"
        ),
    ))

configurar_replay(
    fig,
    f"Votos adicionados em cada atualização — {local}",
    "Votos adicionados no lote",
)
fig.update_yaxes(tickformat=",.0f")
fig.show(config=CONFIG_GRAFICO)

## Síntese automática

In [9]:
lider_por_instante = acumulado.idxmax(axis=1)
trocas_lideranca = int(lider_por_instante.ne(lider_por_instante.shift()).sum() - 1)
lider_final = resultado.index[0]
margem_final = int(resultado.iloc[0, 0] - resultado.iloc[1, 0]) if len(resultado) > 1 else 0
picos = pd.DataFrame({
    "horário_da_atualização": incrementos.idxmax(),
    "votos_adicionados": incrementos.max().astype("int64"),
})

print(f"Líder ao final da seleção: {lider_final}")
print(f"Margem final entre os dois primeiros: {margem_final:,} votos".replace(",", "."))
print(f"Trocas de liderança observadas entre atualizações: {trocas_lideranca}")
print("\nAtualização de maior entrada de votos por candidato:")
display(picos)
con.close()

Líder ao final da seleção: LULA (13)
Margem final entre os dois primeiros: 2.139.645 votos
Trocas de liderança observadas entre atualizações: 4

Atualização de maior entrada de votos por candidato:


,horário_da_atualização,votos_adicionados
candidato,,
JAIR BOLSONARO (22),2022-10-30 17:51:39,43822
LULA (13),2022-10-30 17:51:39,47721


## Limitações

- A fonte informa recebimento com precisão de segundos, mas não uma sequência interna
  para boletins com o mesmo horário.
- Neste notebook, uma atualização é cada valor distinto de `DT_BU_RECEBIDO`. Os
  arquivos de boletim não registram as chamadas ou atualizações visuais do painel
  público do TSE.
- Existem alertas temporais documentados em `04_bi/auditoria/alertas_temporais.csv`.
- O notebook preserva os horários informados e não tenta corrigi-los por fuso ou por
  comparação com `DT_EMISSAO_BU`.